# ViFinQA - Canonical retrieval Qwen rerun

**Kaggle settings:** Accelerator = GPU T4 x2, Internet = On. Attach only dataset `kaggle-payload-canonical-hybrid-w010`.

This run keeps the Qwen 2.5 Coder 7B inference settings used by the previous baseline and changes the retrieval payload only.

In [ ]:
import json, pathlib

PAYLOAD = "/kaggle/input/datasets/kien2005/kaggle-payload-canonical-hybrid-w010"
retrieval_path = pathlib.Path(PAYLOAD) / "retrieval.jsonl"
assert retrieval_path.exists(), f"Missing canonical retrieval: {retrieval_path}"
manifest_path = pathlib.Path(PAYLOAD) / "payload-manifest.json"
assert manifest_path.exists(), f"Missing payload manifest: {manifest_path}"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert manifest.get("schema_version") == 2, manifest.get("schema_version")
assert len(manifest.get("files", {})) == 247, "Unexpected payload; attach the canonical dataset only"
assert sum(1 for _ in retrieval_path.open(encoding="utf-8")) == 1012
print("PAYLOAD =", PAYLOAD, "| verified files =", len(manifest["files"]))

import torch
print("GPUs:", torch.cuda.device_count(), torch.cuda.get_device_name(0))

In [ ]:
# Copy exactly the code fingerprinted in the payload manifest.
import pathlib, shutil

SRC = pathlib.Path(PAYLOAD) / "code"
DST = pathlib.Path("/kaggle/working/code")
shutil.rmtree(DST, ignore_errors=True)
shutil.copytree(SRC, DST)
print("code ->", DST)

In [ ]:
%%time
!pip install -q "transformers>=4.45,<5" "accelerate>=1,<2" "bitsandbytes>=0.45,<1"
import transformers, bitsandbytes
print("transformers", transformers.__version__, "bitsandbytes", bitsandbytes.__version__)

In [ ]:
%%time
# Runtime/OOM smoke test. This output is not used for submission.
!python /kaggle/working/code/kaggle_codegen.py --payload $PAYLOAD --backend hf \
    --model Qwen/Qwen2.5-Coder-7B-Instruct --load-4bit \
    --out /kaggle/working/codegen_canonical_smoke.jsonl --limit 12 \
    --n 1 --temperature 0 --k 4 --max-tokens 256 --batch-size 4 \
    --checkpoint-every 4 --time-budget-min 30 --seed 13 --no-resume

In [ ]:
import collections, json

rows = [json.loads(line) for line in open("/kaggle/working/codegen_canonical_smoke.jsonl", encoding="utf-8")]
print("rows", len(rows), collections.Counter(r["source"] for r in rows))
print("signatures", {r.get("run_signature", "")[:16] for r in rows})
for row in rows[:5]:
    print(row["id"], row["source"], row["answer"], row["question"][:70])

The full run below writes to a new filename. Its semantic settings match the previous 7B select-mode run; the canonical payload changes the shortlist and prompt evidence.

In [ ]:
%%time
!python /kaggle/working/code/kaggle_codegen.py --payload $PAYLOAD --backend hf \
    --model Qwen/Qwen2.5-Coder-7B-Instruct --load-4bit \
    --llm-mode select --llm-target all \
    --out /kaggle/working/codegen_canonical_sel7b.jsonl \
    --n 1 --k 4 --max-tokens 96 --batch-size 8 \
    --checkpoint-every 32 --time-budget-min 400 --seed 13

In [ ]:
# Final integrity checks before downloading the Kaggle output.
import collections, json, math, pathlib

out = pathlib.Path("/kaggle/working/codegen_canonical_sel7b.jsonl")
rows = [json.loads(line) for line in out.open(encoding="utf-8")]
ids = [row["id"] for row in rows]
assert len(rows) == 1012 and len(set(ids)) == 1012, (len(rows), len(set(ids)))
assert set(ids) == set(range(1, 1013))
assert all(math.isfinite(float(row["answer"])) for row in rows)
signatures = {row.get("run_signature", "") for row in rows if row.get("run_signature")}
assert len(signatures) <= 1, signatures
print(collections.Counter(row["source"] for row in rows))
print("OK: 1012 unique finite results ->", out)

## Output

Download `codegen_canonical_sel7b.jsonl` from `/kaggle/working`. Keep the raw file unchanged; it will be audited and merged locally against the confirmed 0.2352 checkpoint before building a submission. Rerunning the full-run cell in the same session resumes records with the matching run signature.